# publish_epss_trend_build
`gold_vuln_daily` 최근 30일 → `publish_epss_trend_30d`(`cve_id`, `date_key`, `epss_score`, `is_systemic_day` 4개 컬럼).
매일 30일을 통째로 재계산하지 않는다 — **오늘 파티션 INSERT + 31일 전 파티션 DELETE**만 한다(03_Gold_Schema.md §3.10 슬라이딩 윈도우). `gold_vuln_daily_build`(DAILY) 완료 후 트리거.

In [ ]:
%run 00_gold_common

In [ ]:
from datetime import datetime as _dt, timedelta as _timedelta

if run_mode == "EVENT":
    raise RuntimeError(
        "publish_epss_trend_build는 EVENT 모드에서 실행하지 않습니다 "
        "(KEV 이벤트는 EPSS 점수를 바꾸지 않음, 05_Job_Orchestration.md §6.4)."
    )

PUBLISH_COLS = ["cve_id", "date_key", "epss_score", "is_systemic_day"]


In [ ]:
if run_mode == "DAILY":
    today_rows = (
        spark.table(TBL["gold_vuln_daily"])
        .filter(F.col("date_key") == snapshot_date)
        .select(*PUBLISH_COLS)
    )
    write_append(today_rows, TBL["publish_epss_trend_30d"], partition_by=["date_key"])

    expire_date = (_dt.strptime(snapshot_date, "%Y-%m-%d") - _timedelta(days=31)).strftime("%Y-%m-%d")
    if table_exists(TBL["publish_epss_trend_30d"]):
        DeltaTable.forName(spark, TBL["publish_epss_trend_30d"]).delete(F.col("date_key") == expire_date)

    written_count = today_rows.count()

elif run_mode in ("BACKFILL", "INITIAL"):
    if run_mode == "BACKFILL" and not target_cve_ids:
        raise ValueError("BACKFILL 모드는 target_cve_ids가 필요합니다.")

    window_start = (_dt.strptime(snapshot_date, "%Y-%m-%d") - _timedelta(days=29)).strftime("%Y-%m-%d")
    bf_rows = spark.table(TBL["gold_vuln_daily"]).filter(
        (F.col("date_key") >= window_start) & (F.col("date_key") <= snapshot_date)
    )
    if run_mode == "BACKFILL":
        bf_rows = bf_rows.filter(F.col("cve_id").isin(target_cve_ids))
    bf_rows = bf_rows.select(*PUBLISH_COLS)

    if table_exists(TBL["publish_epss_trend_30d"]):
        target = DeltaTable.forName(spark, TBL["publish_epss_trend_30d"])
        (
            target.alias("t")
            .merge(bf_rows.alias("s"), "t.cve_id = s.cve_id AND t.date_key = s.date_key")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        write_overwrite(bf_rows, TBL["publish_epss_trend_30d"], partition_by=["date_key"])

    written_count = bf_rows.count()
else:
    raise ValueError(f"Unsupported run_mode: {run_mode}")

log_batch_run("publish_epss_trend_build", "SUCCESS", record_count=written_count)
print(f"[publish_epss_trend_build] mode={run_mode} rows={written_count} batch_id={batch_id}")
